# ImageNet Training Loop

In this notebook, we will run an ImageNet training loop on a single GPU in AWS. Eventually we will create a .py file to train ImageNet on multiple GPUs in AWS or RunPod.

In [1]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../..'))
import time

In [2]:
import torch
import torchvision
from torch.optim import SGD
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.profiler import profile, ProfilerActivity, record_function, schedule

from utils.imagenet import get_train_transform, get_val_transform
from utils.metrics import accuracy, topk_accuracy

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(device, torch.cuda.device_count())

cuda 1


# Hyperparameters

In [4]:
# Mount network volume
bucket_path = '/workspace/imagenet'

# path to save model
save_path = os.getcwd()

# single epoch for testing. Eventually will set to 90
num_epochs = 1

# may need to be smaller if OOM occurs
batch_size = 256

# Change depending on number of CPUs, optimize
num_workers = 24
pin_memory = True
persistent_workers=False

# How many batches before logging loss
log_every = 500

# How many epochs before checkpointing model
checkpoint_every = 2

# How many times to perform validation per epoch
val_per_epoch = 2

# Optimizer hyperparameters
opt_kwargs = {'lr': 0.1 * batch_size/256, 'momentum': 0.9}
num_warmup = 5
T_max = 90
eta_min = 1e-5

# Fixed for ImageNet Dataset
C = 3
H, W = 224, 224
num_classes = 1000

# Import dataset

In [5]:
t1 = time.time()
val_ds = torchvision.datasets.ImageFolder(bucket_path + "/val", transform=get_val_transform())
t2 = time.time()
print(f"Time to create validation dataset: {t2-t1}")

Time to create validation dataset: 1.8696575164794922


In [6]:
t1 = time.time()
train_ds = torchvision.datasets.ImageFolder(bucket_path + "/train", transform=get_train_transform())
t2 = time.time()
print(f"Time to create training dataset: {t2-t1}")

Time to create training dataset: 7.461827039718628


# Create dataloader

In [7]:
train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, 
    num_workers=num_workers, pin_memory=pin_memory, drop_last=True,
    persistent_workers=persistent_workers,
)
print(f"Length of training dataloader is {len(train_dl)}")

Length of training dataloader is 5004


In [8]:
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=2*batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, drop_last=False,
    persistent_workers=persistent_workers,
)
print(f"Length of validation dataloader is {len(val_dl)}")

Length of validation dataloader is 98


# Function to checkpoint model

In [14]:
def save_checkpoint(model, optimizer, epoch, train_records, val_records, path=save_path):
    checkpoint = {
        'epochs': epoch+1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_records': torch.tensor(train_records),
        'val_records': torch.tensor(val_records)
    }
    torch.save(checkpoint, path + '/imagenet-checkpoint.pt')
    print(f"Checkpoint saved after {epoch+1} epochs")

# Write functions for training loop

In [15]:
def is_log_step(i):
    return i % log_every == 0 and i > 0

def is_validate_step(i):
    return (i * val_per_epoch) % len(train_dl) < val_per_epoch

def is_checkpoint_epoch(epoch):
    return epoch % checkpoint_every == 0 and epoch > 0

In [16]:
def compute_metrics(metric_fns, running_metrics, logits, y, reduction='mean'):
    for metric_fn, running_metric in zip(metric_fns, running_metrics):
        running_metric += metric_fn(logits, y, reduction=reduction)

def log_metrics(records, running_metrics, div=1):
    log(records, running_metrics, div=div)
    reset_metric(running_metrics)

def log(records, metrics, div=1):
    for record, metric in zip(records, metrics):
        record.append(metric.item() / div)

def reset_metric(running_metrics):
    for tensor in running_metrics:
        tensor.fill_(0.0)

In [17]:
def train_step(model, X, y, opt, scaler):
    with torch.autocast(device.type):
        logits = model(X)
        
    loss = loss_fn(logits, y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return logits

def val_step(model, X):
    with torch.autocast(device.type):
        logits = model(X)
    return logits

# Introduce loss and training metrics

In [10]:
loss_fn = torch.nn.functional.cross_entropy
metric_fns = [loss_fn, accuracy, topk_accuracy]

# Define model, optimizer, learning rate scheduler, and automatic mixed-precision (AMP) scaler

In [ ]:
def get_new_model():
    return torchvision.models.resnet50().to(device)

In [ ]:
def get_optimizer(model):
    return SGD(model.parameters(), **opt_kwargs)

In [ ]:
def get_scaler():
    return torch.amp.GradScaler(device.type)

In [ ]:
def get_scheduler():
    scheduler1 = LinearLR(opt, 0.01, 1.0, num_warmup)
    scheduler2 = CosineAnnealingLR(opt, T_max=T_max-num_warmup, eta_min=eta_min)
    return SequentialLR(opt, schedulers=[scheduler1, scheduler2], milestones=[num_warmup])

# Profile model

In [18]:
def training_epoch_to_profile(model, opt, scaler):
    running_metrics = [torch.tensor(0.0, device=device) for _ in metric_fns]

    for i, (X, y) in enumerate(train_dl):
        X = X.to(device, non_blocking=pin_memory)
        y = y.to(device, non_blocking=pin_memory)

        if is_log_step(i):
            print(f"Iteration {i}")
    
        logits = train_step(model, X, y, opt, scaler)

        with torch.no_grad():
            compute_metrics(metric_fns, running_metrics, logits, y)

        if i > num_workers * 5:
            break

In [19]:
def trace_handler(prof):
    print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=10))

In [20]:
activities = [ProfilerActivity.CPU, ProfilerActivity.CUDA]

with torch.profiler.profile(
    activities=activities,
    schedule=schedule(wait=1, warmup=1, active=1, skip_first=0, repeat=0),
    on_trace_ready=trace_handler
) as prof:
    model_prof = get_new_model()
    opt_prof = get_optimizer(model_prof)
    scaler_prof = get_scaler()
    for epoch in range(3):
        print(f"Epoch: {epoch}")
        training_epoch_to_profile(model_prof, opt_prof, scaler_prof)
        prof.step()

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  cudaStreamSynchronize        39.17%       14.627s        39.18%       14.628s     117.025ms       0.000us         0.00%       0.000us       0.000us           125  
                                          ProfilerStep*        24.87%        9.284s        91.82%       34.282s       34.282s       0.000us         0.00%        8.105s        8.105s             1  
enumerate

In [21]:
prof.export_chrome_trace("trace.json")

# Create model, optimizer, scheduler, and AMP scaler

In [9]:
model = get_new_model()

In [11]:
opt = get_optimizer(model)

In [12]:
scheduler = get_scheduler()

In [13]:
scaler = get_scaler()

# Run training loop

In [22]:
train_records = [[] for _ in range(len(metric_fns))]
val_records = [[] for _ in range(len(metric_fns))]
running_train_metrics = [torch.tensor(0.0, device=device) for _ in metric_fns]
running_val_metrics = [torch.tensor(0.0, device=device) for _ in metric_fns]

t1 = time.time()
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    for i, (X, y) in enumerate(train_dl):
        X = X.to(device, non_blocking=pin_memory)
        y = y.to(device, non_blocking=pin_memory)

        if is_log_step(i):
            log_metrics(train_records, running_train_metrics, div=log_every)
            print(f"Train loss & accuracy at iter {i}/{len(train_dl)}: "
                   f"{train_records[0][-1]:.5f}, {train_records[1][-1]:.5f}")
        
        logits = train_step(model, X, y, opt, scaler)

        with torch.no_grad():
            compute_metrics(metric_fns, running_train_metrics, logits, y)

        if is_validate_step(i):
            with torch.no_grad():
                for j, (X, y) in enumerate(val_dl):
                    X = X.to(device, non_blocking=pin_memory)
                    y = y.to(device, non_blocking=pin_memory)
                    logits = val_step(model, X)
                    compute_metrics(metric_fns, running_val_metrics, logits, y, reduction='sum')
            log_metrics(val_records, running_val_metrics, div=len(val_ds))
            print(f"Val loss & accuracy at iter {i}/{len(train_dl)}: "
                f"{val_records[0][-1]:.5f}, {val_records[1][-1]:.5f}")

    scheduler.step()

    if is_checkpoint_epoch(epoch):
        save_checkpoint(model, opt, epoch, train_records, val_records)
t2 = time.time()
print(f"Time to run {num_epochs} epochs: {t2-t1:.2f}")

Epoch 1/1
Val loss & accuracy at iter 0/5004: 6.93288, 0.00144
Train loss & accuracy at iter 500/5004: 6.85997, 0.00246
Train loss & accuracy at iter 1000/5004: 6.80877, 0.00411
Train loss & accuracy at iter 1500/5004: 6.75512, 0.00564
Train loss & accuracy at iter 2000/5004: 6.68357, 0.00742
Train loss & accuracy at iter 2500/5004: 6.59412, 0.00966
Val loss & accuracy at iter 2502/5004: 6.85144, 0.00566
Train loss & accuracy at iter 3000/5004: 6.50309, 0.01249
Train loss & accuracy at iter 3500/5004: 6.41356, 0.01498
Train loss & accuracy at iter 4000/5004: 6.31755, 0.01742
Train loss & accuracy at iter 4500/5004: 6.22641, 0.02108
Train loss & accuracy at iter 5000/5004: 6.13229, 0.02316
Time to run 1 epochs: 1166.19


In [23]:
# finally, save the model and metrics
save_checkpoint(model, opt, epoch, train_records, val_records)

Checkpoint saved after 1 epochs
